In [1]:
import pandas as pd 
pd.set_option('display.max_colwidth', None)

In [2]:
import mlflow
import mlflow.sklearn
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

from src.config_mlflow import setup_mlflow

setup_mlflow()

MLflow connected successfully


In [3]:
mlflow.set_experiment("ML_Model_Ticket_Classification(Dataset2)")

2026/06/06 19:20:37 INFO mlflow.tracking.fluent: Experiment with name 'ML_Model_Ticket_Classification(Dataset2)' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/rishabh/Desktop/Customer Support Intelligence/mlruns/3', creation_time=1780753837990, experiment_id='3', last_update_time=1780753837990, lifecycle_stage='active', name='ML_Model_Ticket_Classification(Dataset2)', tags={}, trace_location=None, workspace='default'>

In [4]:
import pandas as pd
import numpy as np
import re
import string
from nltk.stem import WordNetLemmatizer

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [5]:
df=pd.read_csv('../data/ticket_classification.csv')

In [11]:
df.isnull().sum()

subject     0
body        0
answer      0
type        0
queue       0
priority    0
dtype: int64

In [10]:
df.shape

(24617, 6)

In [ ]:
df.head()

In [9]:
df=df.dropna()

In [12]:

def clean_text(text):
    lemmatizer= WordNetLemmatizer()
    stop_words=set(stopwords.words('english'))
    # Lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords + lemmatization
    cleaned_words = []

    for word in words:

        # Remove stopwords
        if word not in stop_words:

            # Lemmatize
            lemma_word = lemmatizer.lemmatize(word)

            cleaned_words.append(lemma_word)

    # Join back
    text = " ".join(cleaned_words)

    return text

In [13]:
df.head()

,subject,body,answer,type,queue,priority
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?","Thank you for reaching out, <name>. We are aware of the outage affecting the centralized account management system, and our technical team is actively working to resolve the issue. In the meantime, we suggest using alternative methods to manage your account, with a focus on restoring service as quickly as possible. We will provide an update as soon as the service is back online. We apologize for the inconvenience and appreciate your patience. If you have any further questions, please let us know.",Incident,Technical Support,high
1,Query About Smart Home System Integration Features,"Dear Customer Support Team,\n\nI hope this message reaches you well. I am reaching out to request detailed information about the capabilities of your smart home integration products listed on your website. As a potential customer aiming to develop a seamlessly interconnected home environment, it is essential to understand how your products interact with various smart home platforms.\n\nCould you kindly provide detailed compatibility information with popular smart home ecosystems such as Amazon Alexa, Google Assistant, and Apple?","Thank you for your inquiry. Our products support integration with Amazon Alexa, Google Assistant, and Apple HomeKit. Compatibility details can differ depending on the specific item; please let us know which models you are interested in. The setup process is generally user-friendly but may require professional installation. We regularly update our software to provide enhanced features. For comprehensive information on compatibility with upcoming updates, please specify the models you are considering.",Request,Returns and Exchanges,medium
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this message finds you well. I am reaching out to request clarification about the billing and payment procedures linked to my account. Recently, I observed some inconsistencies in the charges applied and would like to ensure I fully understand the billing cycle, accepted payment options, and any potential extra charges.\n\nFirstly, I would be grateful if you could provide a detailed explanation of how the billing cycle functions. Specifically, I am interested in knowing the start and end dates.\n\nThank you for your assistance regarding these billing inquiries.","We appreciate you reaching out with your billing questions. The billing period generally begins on the first day of the month and concludes on the last day, with payments due by the 10th of the following month. We accept credit cards, bank transfers, and certain online payment services; credit card transactions are typically processed the quickest. Late payments may incur fees based on the due date, and any additional processing charges depend on the chosen payment method. You can review your statements for detailed payment information.",Request,Billing and Payments,low
3,Question About Marketing Agency Software Compatibility,"Dear Support Team,\n\nI hope this message reaches you well. I am reaching out to ask about the compatibility of your products with the specific needs of marketing agencies. Our company is considering adopting these solutions to streamline our current marketing processes and wants to confirm that the products are fully compatible with the tools and platforms we currently utilize.\n\nCould you please supply detailed information regarding t

In [14]:
df.columns

Index(['subject', 'body', 'answer', 'type', 'queue', 'priority'], dtype='object')

In [15]:
df['Full Text'] = df['subject'].apply(clean_text)+" "+df['body'].apply(clean_text)

# Let's see what a row looks like before and after
print("BEFORE:\n", df['body'].iloc[0])
print("\nAFTER:\n", df['Full Text'].iloc[0])

BEFORE:
 Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

AFTER:
 account disruption dear customer support teamnni writing report significant problem centralized account management portal currently appears offline outage blocking access account setting leading substantial inconvenience attempted log multiple time using different browser device issue persistsnncould please provide update outage status estimated time resolution also alternative way access manage account downtime


In [16]:
from sklearn.preprocessing import LabelEncoder
target_encoder = LabelEncoder()

# Fit and transform your 'ticket type' column into numerical integers
df['Target_Encoded'] = target_encoder.fit_transform(df['queue'])

# Let's see the mapping to understand what it did
mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))
print("Category Mapping Reference:")
print(mapping)

Category Mapping Reference:
{'Billing and Payments': 0, 'Customer Service': 1, 'General Inquiry': 2, 'Human Resources': 3, 'IT Support': 4, 'Product Support': 5, 'Returns and Exchanges': 6, 'Sales and Pre-Sales': 7, 'Service Outages and Maintenance': 8, 'Technical Support': 9}


In [17]:
X = df['Full Text']
y = df['Target_Encoded']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
random_state=42,stratify=y)

In [19]:
tfidf = TfidfVectorizer(max_features=10000,ngram_range=(1,2),
                        min_df=2,
                        max_df=0.95)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [20]:
X_train_tfidf=X_train_tfidf.astype('float32', copy=False)
X_test_tfidf=X_test_tfidf.astype('float32', copy=False)

In [23]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
model_name='Logistic Regression'
with mlflow.start_run(run_name=model_name):
    # Log training dataset metadata (features, targets, predictions link)
    dataset = mlflow.data.from_pandas(
        df, targets="queue", name="dataset"
    )
    mlflow.log_input(dataset, context="Entire Dataset")
    log=LogisticRegression(
        max_iter=2000,
        class_weight='balanced'
    )
    log.fit(X_train_tfidf,y_train)
    predictions=log.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
    mlflow.log_param("max_iter", 2000)
    mlflow.log_param("vectorizer", "TF-IDF")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("TF-IDF", """max_features=10000,ngram_range=(1,2),
                        min_df=2,
                        max_df=0.95""")
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)

    signature = mlflow.models.infer_signature(X_test, predictions)
    # Save model artifact
    mlflow.sklearn.log_model(
        sk_model=log,
        artifact_path=model_name,
        signature=signature,
        input_example=X_test.head(3).to_frame(),
    )

    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score (Weighted): {f1_weighted:.4f}")
    print(f"F1-Score (Macro): {f1_macro:.4f}\n")

/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/06 19:45:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 19:45:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format require

Accuracy: 0.5018
F1-Score (Weighted): 0.5038
F1-Score (Macro): 0.4975

🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/3/runs/5a86e4eb7319401aa56b394fca024a1d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [25]:
model_name='Decision Tree1'
with mlflow.start_run(run_name=model_name):
    # Log training dataset metadata (features, targets, predictions link)
    dataset = mlflow.data.from_pandas(
        df, targets="queue", name="dataset"
    )
    mlflow.log_input(dataset, context="Entire Dataset")
    dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
    dt_model.fit(X_train_tfidf, y_train)
        
        # Predict
    predictions = dt_model.predict(X_test_tfidf)
    text_predictions = target_encoder.inverse_transform(predictions)
    text_y_test = target_encoder.inverse_transform(y_test)
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
    print(f"---Results ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score (Weighted): {f1_weighted:.4f}")
    print(f"F1-Score (Macro): {f1_macro:.4f}\n")
    print(classification_report(text_y_test, text_predictions, zero_division=0))
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("vectorizer", "TF-IDF")
    
    mlflow.log_param("TF-IDF", """max_features=10000,ngram_range=(1,2),
                        min_df=2,
                        max_df=0.95""")
    mlflow.log_param("random_state",42)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    signature = mlflow.models.infer_signature(X_test, predictions)
    # Save model artifact
    mlflow.sklearn.log_model(
        sk_model=log,
        artifact_path=model_name,
        signature=signature,
        input_example=X_test.head(3).to_frame(),
    )

/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/06 19:49:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 19:49:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format require

---Results ---
Accuracy: 0.3954
F1-Score (Weighted): 0.3269
F1-Score (Macro): 0.2433

                                 precision    recall  f1-score   support

           Billing and Payments       0.94      0.62      0.75       509
               Customer Service       0.31      0.20      0.25       751
                General Inquiry       0.00      0.00      0.00        67
                Human Resources       0.75      0.03      0.06        94
                     IT Support       0.50      0.01      0.02       577
                Product Support       0.31      0.14      0.20       928
          Returns and Exchanges       0.81      0.05      0.10       243
            Sales and Pre-Sales       0.80      0.03      0.06       138
Service Outages and Maintenance       0.88      0.35      0.50       199
              Technical Support       0.35      0.88      0.50      1418

                       accuracy                           0.40      4924
                      macro avg     

/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
2026/06/06 19:49:58 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "F.... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: could not convert string to float: 'indepth detail data analytics solution inquiring data analytics solution optimize investment strategy could provide detailed information tool technique used analyze market trend make prediction also

🏃 View run Decision Tree1 at: http://127.0.0.1:5000/#/experiments/3/runs/4c9040da669f4d3e97c9592845211ab4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [26]:
from sklearn.naive_bayes import MultinomialNB

model_name='NaiveBayes_Multinominal'
with mlflow.start_run(run_name=model_name):
    # Log training dataset metadata (features, targets, predictions link)
    dataset = mlflow.data.from_pandas(
        df, targets="queue", name="dataset"
    )
    mlflow.log_input(dataset, context="Entire Dataset")
    nb_model = MultinomialNB(alpha=0.01)
    nb_model.fit(X_train_tfidf, y_train)
        
    # Predict
    predictions = nb_model.predict(X_test_tfidf)
    text_predictions = target_encoder.inverse_transform(predictions)
    text_y_test = target_encoder.inverse_transform(y_test)
        
        # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score (Weighted): {f1_weighted:.4f}")
    print(f"F1-Score (Macro): {f1_macro:.4f}\n")
    print(classification_report(text_y_test, text_predictions, zero_division=0))
    mlflow.log_param("alpha",0.01)
    mlflow.log_param("vectorizer", "TF-IDF")
    
    mlflow.log_param("TF-IDF", """max_features=10000,ngram_range=(1,2),
                        min_df=2,
                        max_df=0.95""")
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    signature = mlflow.models.infer_signature(X_test, predictions)
    # Save model artifact
    mlflow.sklearn.log_model(
        sk_model=nb_model,
        artifact_path=model_name,
        signature=signature,
        input_example=X_test.head(3).to_frame(),
    )

/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/06 19:53:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 19:53:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format require

Accuracy: 0.4773
F1-Score (Weighted): 0.4658
F1-Score (Macro): 0.4488

                                 precision    recall  f1-score   support

           Billing and Payments       0.85      0.69      0.76       509
               Customer Service       0.32      0.46      0.37       751
                General Inquiry       0.93      0.21      0.34        67
                Human Resources       1.00      0.24      0.39        94
                     IT Support       0.57      0.23      0.33       577
                Product Support       0.46      0.30      0.37       928
          Returns and Exchanges       0.79      0.20      0.32       243
            Sales and Pre-Sales       0.44      0.38      0.41       138
Service Outages and Maintenance       0.76      0.57      0.65       199
              Technical Support       0.45      0.70      0.55      1418

                       accuracy                           0.48      4924
                      macro avg       0.66      0.4

/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but MultinomialNB was fitted without feature names
  warnings.warn(
2026/06/06 19:53:09 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "F.... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: could not convert string to float: 'indepth detail data analytics solution inquiring data analytics solution optimize investment strategy could provide detailed information tool technique used analyze market trend make prediction also woul

🏃 View run NaiveBayes_Multinominal at: http://127.0.0.1:5000/#/experiments/3/runs/6672cfa8ac4045a8b747657137f7ab57
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [38]:

from xgboost import XGBClassifier

objective='multi:softprob'
eval_metric='mlogloss'
random_state=42
max_depth=10
n_jobs=-1
n_estimators=250
reg_lambda=5
reg_alpha=0
model_name='XGBclassifier1'
with mlflow.start_run(run_name=model_name):
    # Log training dataset metadata (features, targets, predictions link)
    dataset = mlflow.data.from_pandas(
        df, targets="queue", name="dataset"
    )
    mlflow.log_input(dataset, context="Entire Dataset")
    xgb_model = XGBClassifier(
    objective=objective,
    eval_metric=eval_metric,
    random_state=random_state,
    max_depth=max_depth,
    n_jobs=-1,
    n_estimators=n_estimators
      )

    xgb_model.fit(X_train_tfidf, y_train)

    predictions = xgb_model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
    mlflow.log_param("objective",objective)
    mlflow.log_param("eval_metric",eval_metric)
    mlflow.log_param("random_state",random_state)
    mlflow.log_param("max_depth",max_depth)
    mlflow.log_param("n_jobs",n_jobs)
    mlflow.log_param("n_estimators",n_estimators)
    mlflow.log_param("vectorizer", "TF-IDF")
    
    mlflow.log_param("TF-IDF", """max_features=10000,ngram_range=(1,2),
                        min_df=2,
                        max_df=0.95""")
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    signature = mlflow.models.infer_signature(X_test, predictions)
    # Save model artifact
    mlflow.sklearn.log_model(
        sk_model=xgb_model,
        artifact_path=model_name,
        signature=signature,
        input_example=X_test.head(3).to_frame(),
    )

    print("=" * 50)
    print("XGBOOST RESULTS")
    print("=" * 50)

    print("Accuracy:",
            accuracy_score(y_test, predictions))

    print("F1 Score:",
            f1_score(y_test, predictions, average='macro'))

    print(classification_report(y_test, predictions))

/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/06 20:37:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 20:37:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format require

XGBOOST RESULTS
Accuracy: 0.6864337936636881
F1 Score: 0.6467163779682212
              precision    recall  f1-score   support

           0       0.95      0.83      0.89       509
           1       0.65      0.63      0.64       751
           2       0.81      0.25      0.39        67
           3       0.93      0.43      0.58        94
           4       0.73      0.50      0.59       577
           5       0.64      0.67      0.65       928
           6       0.90      0.45      0.60       243
           7       0.86      0.50      0.63       138
           8       0.92      0.68      0.78       199
           9       0.61      0.85      0.71      1418

    accuracy                           0.69      4924
   macro avg       0.80      0.58      0.65      4924
weighted avg       0.71      0.69      0.68      4924

🏃 View run XGBclassifier1 at: http://127.0.0.1:5000/#/experiments/3/runs/fad6f7e6fa9d47b8889475b877539326
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [37]:
from lightgbm import LGBMClassifier
model_name='LGBMclassifier'
with mlflow.start_run(run_name=model_name):
    # Log training dataset metadata (features, targets, predictions link)
    dataset = mlflow.data.from_pandas(
        df, targets="queue", name="dataset"
    )
    mlflow.log_input(dataset, context="Entire Dataset")
    lgbm_model = LGBMClassifier(
      objective='multiclass',
      random_state=42
      )

    lgbm_model.fit(X_train_tfidf, y_train)

    predictions = lgbm_model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)



    print("Accuracy:",
      accuracy_score(y_test, predictions))

    print("F1 Score:",
      f1_score(y_test, predictions, average='macro'))

    print(classification_report(y_test, predictions))
    mlflow.log_param("objective",'multiclass')
    mlflow.log_param("vectorizer", "TF-IDF")
    
    mlflow.log_param("TF-IDF", """max_features=10000,ngram_range=(1,2),
                        min_df=2,
                        max_df=0.95""")
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    signature = mlflow.models.infer_signature(X_test, predictions)
    # Save model artifact
    mlflow.sklearn.log_model(
        sk_model=lgbm_model,
        artifact_path=model_name,
        signature=signature,
        input_example=X_test.head(3).to_frame(),
    )

/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.771875 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 248152
[LightGBM] [Info] Number of data points in the train set: 19693, number of used features: 6727
[LightGBM] [Info] Start training from score -2.270751
[LightGBM] [Info] Start training from score -1.880319
[LightGBM] [Info] Start training from score -4.297032
[LightGBM] [Info] Start training from score -3.963763
[LightGBM] [Info] Start training from score -2.144749
[LightGBM] [Info] Start training from score -1.668692
[LightGBM] [Info] Start training from score -3.007634
[LightGBM] [Info] Start training from score -3.570854
[LightGBM] [Info] Start training from score -3.207164
[LightGBM] [Info] Start training from score -1.244721


/Users/rishabh/miniconda3/envs/DL_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/06/06 20:30:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/06 20:30:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.6255077173030057
F1 Score: 0.6079332026965465
              precision    recall  f1-score   support

           0       0.91      0.82      0.86       509
           1       0.57      0.53      0.55       751
           2       0.95      0.30      0.45        67
           3       0.95      0.43      0.59        94
           4       0.68      0.40      0.50       577
           5       0.57      0.54      0.55       928
           6       0.83      0.44      0.57       243
           7       0.81      0.46      0.59       138
           8       0.88      0.65      0.75       199
           9       0.55      0.83      0.66      1418

    accuracy                           0.63      4924
   macro avg       0.77      0.54      0.61      4924
weighted avg       0.66      0.63      0.62      4924



2026/06/06 20:30:58 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "F.... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the model uri and serving input example. A serving input example can be generated from model input example using `mlflow.models.convert_input_example_to_serving_input` function.
Got error: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: Full Text: object


🏃 View run LGBMclassifier at: http://127.0.0.1:5000/#/experiments/3/runs/87407d43822c446e8231d0bf574cf819
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
